# 02 — Feature Engineering

Walks through every feature in `src/features.py`, shows its value on benign vs. tunneling examples, and inspects the resulting feature matrix:

- Per-feature value on a sample query (sanity check)
- Per-class distribution of every feature
- Correlation heatmap between features

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.features import FEATURE_FUNCS, extract_features
from src.preprocess import load_dataset

sns.set_theme(style="whitegrid")

## Sanity check on two example queries

In [ ]:
benign = "mail.google.com"
tunneling = "aXk1bG9yZW0gaXBzdW0gZG9sb3Igc2l0.tunnel.evil.com"

rows = []
for name, func in FEATURE_FUNCS.items():
    rows.append({"feature": name, "benign": func(benign), "tunneling": func(tunneling)})
pd.DataFrame(rows).set_index("feature")

Every length / entropy / digit-ratio feature is larger for the tunneling sample — exactly the signal a classifier can exploit.

## Build the full feature matrix

In [ ]:
DATA_PATH = ROOT / "data" / "raw" / "sample.csv"
data, labels = load_dataset(DATA_PATH)

if "query" in data.columns:
    X = extract_features(data["query"])
else:
    X = data

X["label"] = labels.values
X.head()

## Distributions by class

In [ ]:
feature_cols = [c for c in X.columns if c != "label"]
fig, axes = plt.subplots(
    nrows=(len(feature_cols) + 2) // 3, ncols=3, figsize=(14, 12)
)
for ax, col in zip(axes.ravel(), feature_cols):
    sns.kdeplot(
        data=X, x=col, hue="label", common_norm=False, ax=ax, palette="Set1"
    )
    ax.set_title(col)
for ax in axes.ravel()[len(feature_cols):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Correlation heatmap

High collinearity between two features is okay for tree-based models (RF / XGBoost) — they handle redundant features gracefully — but it is worth knowing about for the report.

In [ ]:
corr = X[feature_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Feature correlation")
plt.show()

## Save the processed features

We persist the engineered features so `train.py` and the modeling notebook can reuse them without recomputing.

In [ ]:
out = ROOT / "data" / "processed" / "features.csv"
out.parent.mkdir(parents=True, exist_ok=True)
X.to_csv(out, index=False)
print(f"saved {out}  ({len(X):,} rows, {len(feature_cols)} features)")